# செலவு கோரிக்கை பகுப்பாய்வு

இந்த நோட்புக் உள்ளூர் ரசீது படங்களிலிருந்து பயணச் செலவுகளை செயலாக்கும் ப்ளகின்களை பயன்படுத்தும் முகவர்கள் எவ்வாறு உருவாக்கப்படுவதை, செலவு கோரிக்கை மின்னஞ்சலை உருவாக்கவும், செலவு தரவை ஒரு பை சார்ட் மூலம் காட்சி படுத்தவும் எவ்வாறு செய்கின்றனர் என்பதைக் காட்டுகிறது. முகவர்கள் பணிச்சூழலைப் பொறுத்து செயல்பாடுகளை தானாகத் தேர்ந்தெடுப்பார்கள்.

படியுகள்:
1. OCR முகவர் உள்ளூர் ரசீது படத்தை செயலாக்கி பயணச் செலவு தரவை எடுத்துக்கொள்கிறது.
2. மின்னஞ்சல் முகவர் செலவு கோரிக்கை மின்னஞ்சலை உருவாக்குகிறது.

### பயணச் செலவுக் கட்டாயத்தின் எடுத்துக்காட்டு:
நீங்கள் வேறு நகரத்தில் உள்ள ஒரு தொழில்முனைவு கூட்டத்துக்காகப் பயணம் செய்யும் ஊழியர் என்றிருத்தல் செய்கின்றீர்கள். உங்கள் நிறுவனம் அனைத்து நியாயமான பயண சார்ந்த செலவுகளையும் திருப்பி வழங்கும் கொள்கையை கொண்டுள்ளது. இங்கே சாத்தியமான பயணச் செலவுகளின் விவரிப்பு:
- போக்குவரத்து:
உங்கள் வீட்டு நகரம் முதல் இலக்கு மாநகரம் வரை சுற்றுச்செலுத்து விமான பறக்கு கட்டணம்.
விமான நிலையத்திற்கு போவும் வரவும் டாக்ஸி அல்லது சேவை-கேட்கும் வாகன சேவைகள்.
இலக்கு மாநகரில் உள்ளூர் போக்குவரத்து (பொது போக்குவரத்து, வாடகை வாகனங்கள் அல்லது டாக்சிகள் போன்றவை).

- தங்குமிடம்:
மூன்று இரவுகள் நடுத்தர தரத்திலான தொழில்முனைவு ஹோட்டலில் தங்கல், கூட்ட நடுவணிக்குட்டை அருகே.

- உணவுகள்:
நிறுவனத்தின் ஒரு நாள் உணவு அனுமதி கொள்கையைப் பொறுத்து காலை உணவு, மதியம் உணவு மற்றும் இரவு உணவுக்காக தினசரி அனுமதிக்கொள்கை.

- வேறு செலவுகள்:
விமான நிலையத்தில் பார்கிங் கட்டணங்கள்.
ஹோட்டலில் இணையதள அணுகல் கட்டணங்கள்.
கை 데ம் அல்லது சிறிய சேவை கட்டணங்கள்.

- ஆவணப்படுத்தல்:
நீங்கள் அனைத்து ரசீதுகளையும் (விமானம், டாக்சி, ஹோட்டல், உணவுகள் போன்றவை) மற்றும் செலவு அறிக்கை பூர்த்தி செய்து திருப்பி கொடுத்தல் அளிக்கிறார்.


## தேவையான பதிவுக்களைக் கொண்டு வாருங்கள்

குறிப்பேட்டுக்கான தேவையான பதிவுகள் மற்றும் தொகுதிகளை கொண்டு வாருங்கள்.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## செலவுக் மாதிரிகள் வரையறு

 தனிநபர் செலவுகளுக்கான ஒரு Pydantic மாதிரியை உருவாக்கவும் மற்றும் பயனர் கேள்வியை அமைவுள்ள செலவுக் தரவாக மாற்ற ExpenseFormatter வகுப்பை உருவாக்கவும்.

 ஒவ்வொரு செலவும் பின்வரும் வடிவத்தில் பிரதிநிதித்துவம் செய்யப்படும்:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## கருவிகளை வரையறு - மின்னஞ்சல் உருவாக்குதல்

செலவுத் தவணையை சமர்ப்பிக்க ஒரு மின்னஞ்சலை உருவாக்கும் கருவி செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இன் `@tool` அலங்காரியை பயன்படுத்துகிறது.
- செலவுகளின் மொத்த தொகையை கணக்கிட்டு விவரங்களை மின்னஞ்சல் உடல் வடிவத்தில் அமைக்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ரசீது படங்களிலிருந்து பயணச் செலவுகளைப் பெறும் கருவி

ரசீது படங்களிலிருந்து பயணச் செலவுகளைப் பெற ஒரு கருவிப் செயல்பாட்டை உருவாக்கவும்.
- இந்த கருவி Microsoft Agent Framework இன் `@tool` அலங்காரியை பயன்படுத்துகிறது.
- இது ரசீது படத்தைப் படித்து, அதனை base64 ஆக குறியாக்கம் செய்து, முகவரியை பகுப்பாய்வு செய்ய முகவருக்கான தரவு URI-வை вернуть செய்கிறது.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## செலவுகளை செயலாக்குதல்

ஏஜண்ட்களை வரையறுத்து, அவற்றை `WorkflowBuilder` யைப் பயன்படுத்தி வரிசைப்படுத்தப்பட்ட பணியழுத்தாணியில் இணைக்கவும்.
- OCR ஏஜண்டு `load_receipt_image` கருவியைப் பயன்படுத்தி ரசீதுப் படத்திலிருந்து கட்டமைக்கப்பட்ட செலவு தரவை எடுத்துக்கொள்கிறது.
- Email ஏஜண்டு எடுத்துக் கொண்ட தரவை எடுத்து `generate_expense_email` கருவியைக் கொண்டு ஒரு தொழில்முறை செலவு கோரிக்கை மின்னஞ்சலை உருவாக்கிறது.
- `add_edge` உடன் `WorkflowBuilder` வரிசைப்படுத்தப்பட்ட குழாய்துளையை உருவாக்குகிறது: OCR ஏஜண்டு → Email ஏஜண்டு.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

வரிசைப்படுத்தப்பட்ட வேலைப்பாடை உருவாக்கி, அதை செயல்படுத்தி ரசீத்ப் படம் செயலாக்கி செலவு கூற்று மின்னஞ்சலை உருவாக்கி இயக்கவும்.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**தவிர்க்குறிப்பு**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையான [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டது. நாங்கள் துல்லியத்திற்காக முயற்சித்தாலும், தானாகும் மொழிபெயர்ப்ப்களில் தவறுகள் அல்லது பிழைகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். இதன் முந்தய மொழியில் உள்ள மூல ஆவணம் அதிகாரப்பூர்வ ஆதாரமாகக் கருதப்பட வேண்டும். அவசரமான தகவலுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கின்றோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்பட்ட எந்தவொரு தவறான புரிதலுக்கும் அல்லது தவறான விளக்கங்களுக்கும் நாங்கள் பொறுப்பேற்கமாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
